In [1]:
import os
import nibabel as nib
import numpy as np
from sklearn.mixture import GaussianMixture

In [2]:
def _estimate_gmm_params(data, n_components, covariance_type='full'):
    """
    Estimates GMM segmentation for given data.
    Returns cluster labels for each voxel in 'data'.
    """
    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type=covariance_type,
        n_init=10,
        random_state=42
    )
    segments = gmm.fit_predict(data.reshape(-1, 1))  # reshape to column vector
    return segments

def _normalize_histogram(image):
    """
    Performs contrast enhancement by histogram normalization.
    Maps image intensities to the range [0, 1].
    """
    if np.max(image) == np.min(image):
        return np.zeros_like(image, dtype=np.float64)
    normalized_image = (image - np.min(image)) / (np.max(image) - np.min(image))
    return normalized_image


In [ ]:
# --- Parameters ---
n_clusters_non_brain = 6
brain_cluster_list = [3, 4, 5, 8, 12, 16]

# Path setup
# input_output_folder = "gmm-labels/"
# cases_txt = "gmm-labels/labels_cases.txt"               # text file with subfolder names
input_output_folder = "gmm-all-labels/"
cases_txt = "gmm-all-labels/scans_all.txt"

# --- Read subfolder names from txt file ---
with open(cases_txt, "r") as f:
    case_names = [line.strip() for line in f if line.strip()]

# --- Loop over selected case folders ---
for case_name in case_names:
    case_path = os.path.join(input_output_folder, case_name)

    image_path = os.path.join(case_path, "image.nii.gz")
    mask_path = os.path.join(case_path, "mask.nii.gz")

    if os.path.isdir(case_path) and os.path.exists(image_path) and os.path.exists(mask_path):
        # Load image and mask
        nii_img = nib.load(image_path)
        image_array = np.asarray(nii_img.get_fdata(), dtype=np.float32)

        # Normalize histogram
        normalized_image = _normalize_histogram(image_array)

        mask_nni = nib.load(mask_path)
        mask = np.asarray(mask_nni.get_fdata(), dtype=np.uint8)

        # Flatten image
        flat_image = normalized_image.reshape(-1)

        # Brain and non-brain indices
        brain_idx = np.where(mask.flatten() > 0)[0]
        non_brain_idx = np.where(mask.flatten() == 0)[0]

        # --- Loop over brain cluster counts ---
        for n_clusters_brain in brain_cluster_list:
            # Run GMM for brain
            brain_pixels = flat_image[brain_idx]
            brain_labels = _estimate_gmm_params(brain_pixels, n_clusters_brain)

            # Run GMM for non-brain
            non_brain_pixels = flat_image[non_brain_idx]
            non_brain_labels = _estimate_gmm_params(non_brain_pixels, n_clusters_non_brain)

            # Offset non-brain labels so they don’t overlap brain labels
            non_brain_labels += n_clusters_brain

            # Reconstruct segmented image
            segmented_flat = np.zeros_like(flat_image, dtype=np.uint8)
            segmented_flat[brain_idx] = brain_labels
            segmented_flat[non_brain_idx] = non_brain_labels
            segmented_image = segmented_flat.reshape(image_array.shape)

            # Save segmented result with cluster count in filename
            save_name = f"labels_gmm_{n_clusters_brain}c.nii.gz"
            save_path = os.path.join(case_path, save_name)
            segmented_nii = nib.Nifti1Image(segmented_image.astype(np.uint8), affine=nii_img.affine)
            nib.save(segmented_nii, save_path)


print("All selected cases processed and saved.")

In [ ]:
import os
import nibabel as nib
import matplotlib.pyplot as plt
import re

# --- 1. Define the Label Mapping ---
label_map = {
    '3': '3 Brain-Specific Labels',
    '4': '4 Brain-Specific Labels',
    '5': '5 Brain-Specific Labels',
    '8': '8 Brain-Specific Labels',
    '12': '12 Brain-Specific Labels',
    '16': '16 Brain-Specific Labels'
}
# Regex pattern to extract cluster number from filename like "labels_gmm_3c.nii.gz"
LABEL_NUMBER_PATTERN = re.compile(r'labels_gmm_(\d+)c\.nii\.gz$')

# --- 2. Main Logic ---
files_path = input_output_folder

for case_name in os.listdir(files_path):
    case_path = os.path.join(files_path, case_name)
    if not os.path.isdir(case_path):
        continue

    print(f"\n--- Showing results for case: {case_name} ---")

    # Get all labels*.nii.gz files in this case folder
    all_label_files = sorted([f for f in os.listdir(case_path) if f.startswith("labels_gmm") and f.endswith(".nii.gz")])

    # Filter files based on your brain_cluster_list
    brain_cluster_list = [3,4,5,8,12,16]
    label_files = []
    for f in all_label_files:
        match = re.search(LABEL_NUMBER_PATTERN, f)
        if match and int(match.group(1)) in brain_cluster_list:
            label_files.append(f)

    # --- Sort files numerically by cluster number ---
    label_files.sort(key=lambda x: int(re.search(LABEL_NUMBER_PATTERN, x).group(1)))

    if not label_files:
        print(f"No matching label files found in {case_path}")
        continue

    # Load the base image (for background)
    image_path = os.path.join(case_path, "image.nii.gz")
    if not os.path.exists(image_path):
        print(f"No image found in {case_path}")
        continue
    nii_img = nib.load(image_path).get_fdata()
    slice_index = nii_img.shape[2] // 2  # middle slice
    base_img = nii_img[:, :, slice_index]

    # Determine subplot grid
    n_labels = len(label_files)
    rows = (n_labels + 3) // 3  # 4 columns, auto rows
    cols = 3

    plt.figure(figsize=(cols * 5, rows * 5))

    # Plot each segmentation
    for i, label_file in enumerate(label_files, start=1):
        label_path = os.path.join(case_path, label_file)
        match = re.search(LABEL_NUMBER_PATTERN, label_file)
        label_key = match.group(1) if match else None
        plot_title = label_map.get(label_key, label_file.replace(".nii.gz", ""))

        seg_data = nib.load(label_path).get_fdata()[:, :, 0]  # first slice for segmentation

        plt.subplot(rows, cols, i)
        plt.imshow(base_img, cmap="gray")
        plt.imshow(seg_data, cmap="nipy_spectral", alpha=0.5)
        plt.title(plot_title, fontsize=24)
        plt.axis("off")

    plt.suptitle(f"GMM Segmentations for {case_name}", fontsize=25)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()